In [1]:
# What happens with device_map="auto" (your current code)
# model = GPT2LMHeadModel.from_pretrained(
#     "openai-community/gpt2",
#     quantization_config=bnb_config,
#     device_map="auto"
# )
# Behind the scenes:
# 🤗 Transformers inspects:
# available GPUs
# GPU memory
# quantization type (4-bit here)
# Then it automatically places the model:
# Mostly on GPU
# Some layers on CPU only if needed
# On Colab T4:
# GPT-2 (4-bit) → entire model fits on GPU
# Result:
# ✔️ Fast
# ✔️ Safe
# ✔️ No OOM
# ✔️ Correct device placement
# This is why your training actually worked.
# What if you remove device_map entirely?
# model = GPT2LMHeadModel.from_pretrained(
#     "openai-community/gpt2",
#     quantization_config=bnb_config
# )
# What happens then?
# By default:
# The model is loaded on CPU
# Quantized weights stay on CPU
# When training starts:
# Inputs go to GPU
# Model stays on CPU
# 💥 Device mismatch error
# Typical error:
# RuntimeError: Expected all tensors to be on the same device
# Even worse:
# GPT-2 + 4bit + LoRA cannot be moved later using .to("cuda")
# BitsAndBytes locks the device at load time
# 👉 So removing device_map usually breaks QLoRA training.

In [2]:
!pip install -q bitsandbytes accelerate datasets peft transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [4]:
import torch
from datasets import load_dataset
import transformers
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from dataclasses import dataclass
from typing import Dict, List
import evaluate
import time

In [5]:
print("Loading dataset...")
dataset = load_dataset("rajpurkar/squad")

# Use subsets for speed
train_dataset = dataset["train"].select(range(3000))
eval_dataset = dataset["validation"].select(range(200))

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [6]:
# train_dataset = dataset["train"]
# len(train_dataset)

In [7]:
# eval_dataset=dataset["validation"]
# len(eval_dataset)

In [8]:
# train_dataset = dataset["train"].select(range(10000))
# eval_dataset = dataset["validation"].select(range(1000))

In [9]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

In [10]:
print("Loading tokenizer and model...")

tokenizer = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Important for causal LM
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2",quantization_config=bnb_config,device_map="auto")
model.config.pad_token_id = tokenizer.eos_token_id

Loading tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
def preprocess(example):
    # ----- Chat-style template (paper style) -----
    model_chat_template = (
        "<|system|>\n"
        "You are a question answer assistant chatbot named \"QA Bot\". "
        "Your expertise is exclusively in providing information and advice "
        "related only to the given content.\n"
        "<|end|>\n\n"
        "<|assistant|>\n"
        "QA Bot here! Ask me anything related to the content you provide.\n"
        "<|end|>\n"
        "<|user|>\n"
    )

    followup_template = (
        "<|assistant|>\n"
        "Ask question regarding this content.\n"
        "<|end|>\n"
        "<|user|>\n"
    )

    context = example["context"]
    question = example["question"]
    answer = example["answers"]["text"][0]

    # ----- Full input text (NO answer included) -----
    input_text = (
        model_chat_template
        + context
        + "<|end|>\n"
        + followup_template
        + question
        + "<|end|>"
    )

    # ----- Tokenize input -----
    input_encodings = tokenizer(
        input_text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None,
        add_special_tokens=False,
    )

    # ----- Tokenize answer separately -----
    answer_encodings = tokenizer(
        answer,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None,
        add_special_tokens=False,
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": answer_encodings["input_ids"],
    }

print("Tokenizing train dataset...")
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

print("Tokenizing eval dataset...")
eval_dataset = eval_dataset.map(
    preprocess,
    remove_columns=dataset["validation"].column_names
)



Tokenizing train dataset...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing eval dataset...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [12]:
print("Preparing QLoRA model...")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
# Disable the use of cache in the PEFT model configuration to reduce memory usage during training
model.config.use_cache = False
model.print_trainable_parameters()

Preparing QLoRA model...
trainable params: 1,622,016 || all params: 126,061,824 || trainable%: 1.2867


In [13]:
# F1 Evaluation Metric
metric = evaluate.load("f1", trust_remote_code = True)
# Compute evaluation metrics
def compute_metrics(eval_pred):
    with torch.no_grad():
      torch.cuda.empty_cache()  # Clear GPU cache
      predictions, labels = eval_pred
      predictions = predictions.argmax(dim=-1)
      true_labels = [label for pred, label in zip(predictions, labels) if label != -100]
      true_preds = [pred for pred, label in zip(predictions, labels) if label != -100]
    return metric.compute(predictions=true_preds, references=true_labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qa-gpt2-lora",
    max_steps=300,
    per_device_train_batch_size=8,  # Reduced from 16
    per_device_eval_batch_size=4,   # Reduced from 16
    gradient_accumulation_steps=2,  # Increased to maintain effective batch size
    # Number of steps to accumulate gradients before performing a backward/update pass
    learning_rate=3e-4,
    warmup_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,  # Log at EVERY step
    logging_first_step=False,
    save_strategy="steps", # Strategy to save the model checkpoint
    eval_strategy="steps",  # Disabled to save memory
    eval_steps=10,
    eval_accumulation_steps = 1,
    report_to="none",
    disable_tqdm=False,
    dataloader_num_workers=2,
    seed=42,
    gradient_checkpointing=True,  # Enable for memory savings
    # do_eval = True,   # Perform evaluation during training to reduce training time
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm = False),
    compute_metrics = compute_metrics
)


print("\n🚀 Starting training...")
start = time.time()
train_result = trainer.train()
duration = (time.time() - start) / 60
print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE")
print(f"⏱ Duration: {duration:.2f} minutes")
print(f"📉 Final loss: {train_result.training_loss:.4f}")
print(f"🔥 Steps: {train_result.global_step}")
print("=" * 60)


🚀 Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
